In [94]:
# Import required libraries

import numpy as np
import pandas as pd
import os
from datetime import date, datetime, timezone
from typing import Literal
%pip install fastapi uvicorn
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel, Field
from sklearn.cluster import KMeans
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, RobustScaler, StandardScaler
from sklearn.model_selection import StratifiedKFold, cross_validate

Note: you may need to restart the kernel to use updated packages.


In [95]:
# Load cleaned_data on pandas

filepath = r"C:\Users\USER\AppData\Local\Packages\5319275A.WhatsAppDesktop_cv1g1gvanyjgm\LocalState\sessions\86DED888A317FE2A325DB1FAFCECA89DDEB8FFD1\transfers\2026-47\Hotel_booking_project_cleaned with Dashboard_(2)(1).xlsx"

,User ID,Room Type/ID,Check-in Date,Check-out Date,Guest Count,Booking Status,Amount Per Night,Total Amount,Booking Started Time,Booking Confirmed Time,...,Payment Reference,Payment Method,Payment Status,Payment Initiation Time,Payment Confirmation Time,Room Type,Room ID,Stay Nights,Booking Duration (min),Payment Duration (min)
0,USR-10814,Deluxe / 220,2026-10-25,2026-10-28,2,Completed,55000,165000,2026-10-17 20:40:00,2026-10-17 21:03:00,...,PAY-USD-000001,Mobile Money,Paid,2026-10-17 20:46:00,2026-10-17 20:58:00,Deluxe,220,3,23.0,12.0
1,USR-10307,Executive / 301,2026-05-18,2026-05-21,2,Completed,80000,240000,2026-04-26 21:00:00,2026-04-26 21:14:00,...,PAY-USD-000002,Bank Transfer,Paid,2026-04-26 21:02:00,2026-04-26 21:05:00,Executive,301,3,14.0,3.0
2,USR-10091,Deluxe / 220,2026-11-13,2026-11-14,4,Completed,55000,55000,2026-11-09 13:40:00,2026-11-09 14:05:00,...,PAY-MM-000003,Card,Paid,2026-11-09 13:55:00,2026-11-09 13:58:00,Deluxe,220,1,25.0,3.0
3,USR-10855,Suite / 404,2026-10-03,2026-10-07,2,Completed,125000,500000,2026-09-04 20:45:00,2026-09-04 21:08:00,...,PAY-TRF-000004,Mobile Money,Paid,2026-09-04 21:01:00,2026-09-04 21:02:00,Suite,404,4,23.0,1.0
4,USR-10708,Deluxe / 205,2026-10-19,2026-10-23,2,Completed,55000,220000,2026-09-15 21:45:00,2026-09-15 22:16:00,...,PAY-MM-000005,USSD,Paid,2026-09-15 22:01:00,2026-09-15 22:06:00,Deluxe,205,4,31.0,5.0


In [96]:
# Feature engineering

df["lead_time_days"] = (
    df["Check-in Date"] - df["Booking Started Time"]
).dt.total_seconds() / 86400.0
df["checkin_dayofweek"] = df["Check-in Date"].dt.dayofweek
df["checkin_month"] = df["Check-in Date"].dt.month
df["is_weekend_checkin"] = df["checkin_dayofweek"].isin([4, 5, 6]).astype(int)
df["booking_start_hour"] = df["Booking Started Time"].dt.hour
df["is_late_night_booking"] = (
    df["booking_start_hour"].between(22, 23) | df["booking_start_hour"].between(0, 5)
).astype(int)
df["payment_share_of_booking"] = np.where(
    df["Booking Duration (min)"] > 0,
    df["Payment Duration (min)"] / df["Booking Duration (min)"],
    0.0,
)
df["user_total_bookings"] = df.groupby("User ID")["Booking ID"].transform("count")
df["is_repeat_guest"] = (df["user_total_bookings"] > 1).astype(int)
df["is_cancelled"] = (df["Booking Status"] == "Cancelled").astype(int)

In [97]:
# Baseline cancellation model (5-fold Stratified CV)

num_cols = [
    "lead_time_days", "Stay Nights", "Guest Count", "Amount Per Night",
    "Total Amount", "Booking Duration (min)", "Payment Duration (min)",
    "payment_share_of_booking", "is_weekend_checkin", "is_late_night_booking",
    "user_total_bookings", "is_repeat_guest",
]
cat_cols = ["Room Type", "Payment Method"]

preprocessor = ColumnTransformer([
    ("num", StandardScaler(), num_cols),
    ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols),
])

clf_pipe = Pipeline([
    ("prep", preprocessor),
    ("model", LogisticRegression(class_weight="balanced", max_iter=1000, random_state=42)),
])

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_results = cross_validate(clf_pipe, df[num_cols + cat_cols], df["is_cancelled"], cv=cv, scoring=["roc_auc", "f1"])

In [98]:
# Fit on full dataset for prototype scoring

clf_pipe.fit(df[num_cols + cat_cols], df["is_cancelled"])
df["cancellation_risk_score"] = clf_pipe.predict_proba(df[num_cols + cat_cols])[:, 1]
df["cancellation_risk_tier"] = pd.cut(
    df["cancellation_risk_score"],
    bins=[-0.01, 0.35, 0.65, 1.01],
    labels=["LOW", "MEDIUM", "HIGH"],
)

In [99]:
# Guest segmentation prototype

seg_cols = ["Stay Nights", "Guest Count", "Amount Per Night", "Total Amount", "lead_time_days"]
seg_scaler = StandardScaler()
seg_kmeans = KMeans(n_clusters=3, n_init=20, random_state=42)
df["segment_id"] = seg_kmeans.fit_predict(seg_scaler.fit_transform(df[seg_cols]))

In [100]:
print(df["segment_id"])

0     1
1     1
2     1
3     0
4     2
     ..
95    1
96    1
97    1
98    1
99    1
Name: segment_id, Length: 100, dtype: int32


In [101]:
###  FASTAPI AND PYDANTIC IMPLEMENTATION
## Save this as guest_experience_api.py and run with uvicorn guest_experience_api:app --reload. It loads Cleaned_Data from Hotel_booking_project_cleaned with Dashboard_(2)(1).xlsx, trains the baseline cancellation and segmentation models at startup, and serves the frontend endpoints.

In [102]:
EXCEL_PATH = r"C:\Users\USER\ppData\Local\Packages\5319275A.WhatsAppDesktop_cv1g1gvanyjgm\LocalState\sessions\86DED888A317FE2A325DB1FAFCECA89DDEB8FFD1\transfers\2026-47\Hotel_booking_project_cleaned with Dashboard_(2)(1).xlsx""
SHEET_NAME = "Cleaned_Data"
MODEL_VERSION = "cancel_logreg_v0.2_reg"
SEGMENT_VERSION = "seg_kmeans_k3_v0.2"

SyntaxError: unterminated string literal (detected at line 1) (2077950426.py, line 1)

In [ ]:
# Pydantic response schemas

class StaySummary(BaseModel):
    room_type: str
    room_id: str
    checkin_date: date
    checkout_date: date
    stay_nights: int
    guest_count: int
    lead_time_days: float


class FinancialSummary(BaseModel):
    amount_per_night: float
    total_amount: float
    currency: str | None = Field(
        default=None,
        description="Null until currency is confirmed in Cleaning_Log / data source.",
    )
    payment_method: str
    payment_reference: str


class CancellationRisk(BaseModel):
    cancellation_risk_score: float = Field(..., ge=0.0, le=1.0)
    cancellation_risk_tier: Literal["LOW", "MEDIUM", "HIGH"]
    model_version: str
    is_directional_estimate: bool = True
    recommended_action: str


class BookingInsightsResponse(BaseModel):
    booking_id: str
    user_id: str
    booking_status: str
    payment_status: str
    stay_summary: StaySummary
    financials: FinancialSummary
    risk: CancellationRisk


class GuestSegmentInfo(BaseModel):
    segment_id: int
    segment_label: Literal[
        "SHORT_STAY_VALUE", "HIGH_SPEND_EXTENDED", "LONG_STAY_BUDGET"
    ]
    segment_version: str
    preferred_room_type: str
    preferred_payment_method: str
    avg_stay_nights: float
    avg_total_amount: float


class UiPersonalization(BaseModel):
    show_repeat_guest_badge: bool
    featured_offer_type: str


class GuestProfileResponse(BaseModel):
    user_id: str
    is_repeat_guest: bool
    total_bookings: int
    segment: GuestSegmentInfo
    ui_personalization: UiPersonalization


class CheckoutMethodMetric(BaseModel):
    payment_method: str
    booking_count: int
    avg_booking_duration_min: float
    avg_payment_duration_min: float
    observed_cancellation_rate: float
    high_friction_flag: bool


class CheckoutHealthResponse(BaseModel):
    as_of: datetime
    is_directional_estimate: bool = True
    methods: list[CheckoutMethodMetric]

In [ ]:
# Data Prep & Model Training

def build_feature_store(excel_path: str, sheet_name: str) -> tuple[pd.DataFrame, pd.DataFrame]:
    df = pd.read_excel(excel_path, sheet_name=sheet_name)
    df["Room ID"] = df["Room ID"].astype(str)

    df["lead_time_days"] = (
        df["Check-in Date"] - df["Booking Started Time"]
    ).dt.total_seconds() / 86400.0
    df["is_long_lead"] = (df["lead_time_days"] >= 30.0).astype(int)
    df["checkin_dayofweek"] = df["Check-in Date"].dt.dayofweek
    df["is_weekend_checkin"] = df["checkin_dayofweek"].isin([4, 5, 6]).astype(int)
    df["booking_start_hour"] = df["Booking Started Time"].dt.hour
    df["is_late_night_booking"] = (
        df["booking_start_hour"].between(22, 23) | df["booking_start_hour"].between(0, 5)
    ).astype(int)
    df["payment_share_of_booking"] = np.where(
        df["Booking Duration (min)"] > 0,
        df["Payment Duration (min)"] / df["Booking Duration (min)"],
        0.0,
    )
    df["log_amount_per_night"] = np.log1p(df["Amount Per Night"])
    df["log_total_amount"] = np.log1p(df["Total Amount"])

    user_counts = df.groupby("User ID")["Booking ID"].transform("count")
    df["user_total_bookings"] = user_counts
    df["is_repeat_guest"] = (df["user_total_bookings"] > 1).astype(int)
    df["is_cancelled"] = (df["Booking Status"] == "Cancelled").astype(int)

    num_cols = [
        "lead_time_days",
        "is_long_lead",
        "Stay Nights",
        "Guest Count",
        "log_amount_per_night",
        "log_total_amount",
        "Payment Duration (min)",
        "payment_share_of_booking",
        "is_weekend_checkin",
        "is_late_night_booking",
        "is_repeat_guest",
    ]
    cat_cols = ["Room Type", "Payment Method"]

    preprocessor = ColumnTransformer([
        ("num", RobustScaler(), num_cols),
        ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols),
    ])
    clf_pipeline = Pipeline([
        ("prep", preprocessor),
        ("model", LogisticRegression(C=0.1, class_weight="balanced", max_iter=1000, random_state=42)),
    ])
    clf_pipeline.fit(df[num_cols + cat_cols], df["is_cancelled"])

    df["cancellation_risk_score"] = np.round(
        clf_pipeline.predict_proba(df[num_cols + cat_cols])[:, 1], 4
    )
    df["cancellation_risk_tier"] = pd.cut(
        df["cancellation_risk_score"],
        bins=[-0.001, 0.35, 0.65, 1.001],
        labels=["LOW", "MEDIUM", "HIGH"],
    ).astype(str)

    seg_cols = [
        "Stay Nights",
        "Guest Count",
        "log_amount_per_night",
        "log_total_amount",
        "lead_time_days",
    ]
    seg_scaler = StandardScaler()
    kmeans = KMeans(n_clusters=3, n_init=30, random_state=42)
    raw_clusters = kmeans.fit_predict(seg_scaler.fit_transform(df[seg_cols]))

In [ ]:
# Map clusters deterministically by average total spend

def create_labels(df, raw_clusters):
    cluster_spend = (
        pd.Series(df["Total Amount"].values)
        .groupby(raw_clusters)
        .mean()
        .sort_values()
    )
    ordered_labels = ["SHORT_STAY_VALUE", "LONG_STAY_BUDGET", "HIGH_SPEND_EXTENDED"]
    cluster_to_label = {
        int(cid): ordered_labels[idx] for idx, cid in enumerate(cluster_spend.index)
    }
    df["segment_id"] = raw_clusters
    df["segment_label"] = df["segment_id"].map(cluster_to_label)

    user_profiles = (
        df.groupby("User ID", as_index=False)
        .agg(
            total_bookings=("Booking ID", "count"),
            is_repeat_guest=("is_repeat_guest", "max"),
            segment_id=("segment_id", lambda s: int(s.mode().iloc[0])),
            segment_label=("segment_label", lambda s: s.mode().iloc[0]),
            preferred_room_type=("Room Type", lambda s: s.mode().iloc[0]),
            preferred_payment_method=("Payment Method", lambda s: s.mode().iloc[0]),
            avg_stay_nights=("Stay Nights", "mean"),
            avg_total_amount=("Total Amount", "mean"),
        )
    )
    return df, user_profiles


bookings_df, users_df = build_feature_store(EXCEL_PATH, SHEET_NAME)
bookings_by_id = bookings_df.set_index("Booking ID")
users_by_id = users_df.set_index("User ID")

app = FastAPI(title="Guest Experience Data Science API", version="0.2.0")


@app.get(
    "/api/v1/guest-experience/bookings/{booking_id}/insights",
    response_model=BookingInsightsResponse,
)
def get_booking_insights(booking_id: str) -> BookingInsightsResponse:
    if booking_id not in bookings_by_id.index:
        raise HTTPException(status_code=404, detail="Booking ID not found")

    row = bookings_by_id.loc[booking_id]
    tier: Literal["LOW", "MEDIUM", "HIGH"] = row["cancellation_risk_tier"]  # type: ignore[assignment]
    action_map = {
        "HIGH": "OFFER_FLEXIBLE_CHECKIN_AND_CONCIERGE_ASSIST",
        "MEDIUM": "SEND_PRE_ARRIVAL_CHECKIN_REMINDER",
        "LOW": "STANDARD_WELCOME_EXPERIENCE",
    }

    return BookingInsightsResponse(
        booking_id=str(booking_id),
        user_id=str(row["User ID"]),
        booking_status=str(row["Booking Status"]),
        payment_status=str(row["Payment Status"]),
        stay_summary=StaySummary(
            room_type=str(row["Room Type"]),
            room_id=str(row["Room ID"]),
            checkin_date=row["Check-in Date"].date(),
            checkout_date=row["Check-out Date"].date(),
            stay_nights=int(row["Stay Nights"]),
            guest_count=int(row["Guest Count"]),
            lead_time_days=round(float(row["lead_time_days"]), 2),
        ),
        financials=FinancialSummary(
            amount_per_night=float(row["Amount Per Night"]),
            total_amount=float(row["Total Amount"]),
            currency=None,
            payment_method=str(row["Payment Method"]),
            payment_reference=str(row["Payment Reference"]),
        ),
        risk=CancellationRisk(
            cancellation_risk_score=float(row["cancellation_risk_score"]),
            cancellation_risk_tier=tier,
            model_version=MODEL_VERSION,
            is_directional_estimate=True,
            recommended_action=action_map[tier],
        ),
    )


@app.get(
    "/api/v1/guest-experience/users/{user_id}/profile",
    response_model=GuestProfileResponse,
)
def get_user_profile(user_id: str): 
    if user_id not in users_by_id.index:
        raise HTTPException(status_code=404, detail="User ID not found")

    u = users_by_id.loc[user_id]
    is_repeat = bool(u["is_repeat_guest"])
    seg_label: Literal[
        "SHORT_STAY_VALUE", "HIGH_SPEND_EXTENDED", "LONG_STAY_BUDGET"
    ] = u["segment_label"]  

    offer_map = {
        "HIGH_SPEND_EXTENDED": "SUITE_UPGRADE_OR_LATE_CHECKOUT_PERK",
        "LONG_STAY_BUDGET": "EXTENDED_STAY_DINING_OR_LAUNDRY_BUNDLE",
        "SHORT_STAY_VALUE": "QUICK_WEEKEND_OR_FLEX_ADDON",
    }

    return GuestProfileResponse(
        user_id=str(user_id),
        is_repeat_guest=is_repeat,
        total_bookings=int(u["total_bookings"]),
        segment=GuestSegmentInfo(
            segment_id=int(u["segment_id"]),
            segment_label=seg_label,
            segment_version=SEGMENT_VERSION,
            preferred_room_type=str(u["preferred_room_type"]),
            preferred_payment_method=str(u["preferred_payment_method"]),
            avg_stay_nights=round(float(u["avg_stay_nights"]), 2),
            avg_total_amount=round(float(u["avg_total_amount"]), 2),
        ),
        ui_personalization=UiPersonalization(
            show_repeat_guest_badge=is_repeat,
            featured_offer_type=offer_map[seg_label],
        ),
    )


@app.get(
    "/api/v1/guest-experience/checkout-health",
    response_model=CheckoutHealthResponse,
)
def get_checkout_health() -> CheckoutHealthResponse:
    grouped = bookings_df.groupby("Payment Method", as_index=False).agg(
        booking_count=("Booking ID", "count"),
        avg_booking_duration_min=("Booking Duration (min)", "mean"),
        avg_payment_duration_min=("Payment Duration (min)", "mean"),
        observed_cancellation_rate=("is_cancelled", "mean"),
    )
    methods: list[CheckoutMethodMetric] = []
    for _, r in grouped.iterrows():
        avg_pay = round(float(r["avg_payment_duration_min"]), 2)
        cancel_rate = round(float(r["observed_cancellation_rate"]), 3)
        methods.append(
            CheckoutMethodMetric(
                payment_method=str(r["Payment Method"]),
                booking_count=int(r["booking_count"]),
                avg_booking_duration_min=round(float(r["avg_booking_duration_min"]), 2),
                avg_payment_duration_min=avg_pay,
                observed_cancellation_rate=cancel_rate,
                high_friction_flag=bool(avg_pay >= 7.0 or cancel_rate >= 0.10),
            )
        )

    return CheckoutHealthResponse(
        as_of=datetime.now(timezone.utc),
        is_directional_estimate=True,
        methods=methods,
    )